# Agentic Youtube agent
### OpenAI Agents SDK

In [15]:
from agents import Agent, function_tool, Runner
import requests
from typing import Optional
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

In [16]:
def get_page_content(url: str) -> Optional[str]:
    """
    Fetch the Markdown content of a web page using the Jina Reader service.

    This function prepends the Jina Reader proxy URL to the provided `url`,
    sends a GET request with a timeout, and decodes the response as UTF-8 text.

    Args:
        url (str): The URL of the page to fetch.

    Returns:
        Optional[str]: The Markdown-formatted content of the page if the request
        succeeds; otherwise, None.

    Raises:
        None: All network or decoding errors are caught and suppressed.
               Logs or error messages could be added as needed.
    """
    reader_url_prefix = "https://r.jina.ai/"
    reader_url = reader_url_prefix + url

    try:
        response = requests.get(reader_url, timeout=10)
        response.raise_for_status()  # raises for 4xx/5xx HTTP errors
        return response.content.decode("utf-8")
    except (requests.exceptions.RequestException, UnicodeDecodeError) as e:
        # Optional: log or print the error for debugging
        print(f"Error fetching content from {url}: {e}")
        return None

In [17]:
assistant_instructions = """
You're a helpful assistant that helps answer user questions.
"""

assistant = Agent(
    name='assistant',
    tools=[function_tool(get_page_content)],
    instructions=assistant_instructions,
    model='gpt-4o-mini'
)

In [18]:
runner = Runner()

In [19]:
user_prompt = "Summarize the content of https://openai.github.io/openai-agents-python/"
result = await runner.run(assistant, input=user_prompt)

In [20]:
print(result.final_output)

The **OpenAI Agents SDK** is designed for developing agentic AI applications with a user-friendly interface. It builds on previous tools like Swarm, providing core components to create robust solutions easily. Key features include:

1. **Agents**: Large Language Models (LLMs) that can use specific instructions and tools.
2. **Handoffs**: Allow agents to delegate tasks to one another.
3. **Guardrails**: Validate inputs and outputs for safety.
4. **Sessions**: Automatically track conversation history.

The SDK simplifies development by offering built-in functionalities such as an agent loop, input validation, and easy integration with Python. It aims to balance comprehensive features with ease of use, making it suitable for various real-world applications.

### Installation
You can install the SDK using:
```bash
pip install openai-agents
```

### Example Usage
A simple example demonstrates how to set up an agent and run a command to generate a haiku about recursion:
```python
from agents

In [21]:
# all agent actions
items = result.new_items

In [22]:
result.new_items[-1].raw_item.content[0].text

'The **OpenAI Agents SDK** is designed for developing agentic AI applications with a user-friendly interface. It builds on previous tools like Swarm, providing core components to create robust solutions easily. Key features include:\n\n1. **Agents**: Large Language Models (LLMs) that can use specific instructions and tools.\n2. **Handoffs**: Allow agents to delegate tasks to one another.\n3. **Guardrails**: Validate inputs and outputs for safety.\n4. **Sessions**: Automatically track conversation history.\n\nThe SDK simplifies development by offering built-in functionalities such as an agent loop, input validation, and easy integration with Python. It aims to balance comprehensive features with ease of use, making it suitable for various real-world applications.\n\n### Installation\nYou can install the SDK using:\n```bash\npip install openai-agents\n```\n\n### Example Usage\nA simple example demonstrates how to set up an agent and run a command to generate a haiku about recursion:\n```

In [23]:
# run the agent interactively
chat_interface = IPythonChatInterface()

runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=assistant
)

In [49]:
await runner.run();

Chat ended.


### YouTube Video Summary Agent

In [24]:
from youtube_transcript_api import YouTubeTranscriptApi

def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"


def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)


def fetch_transcript_raw(video_id):
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(video_id)
    return transcript


def fetch_transcript_text(video_id):
    transcript = fetch_transcript_raw(video_id)
    subtitles = make_subtitles(transcript)
    return subtitles  

In [25]:
from pathlib import Path

def fetch_transcript_cached(video_id):
    cache_dir = Path("../data_cache/youtube_videos")
    cache_file = cache_dir / f"{video_id}.txt"

    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8")

    subtitles = fetch_transcript_text(video_id)
    cache_file.write_text(subtitles, encoding="utf-8")

    return subtitles

In [26]:
def fetch_youtube_transcript(video_id: str) -> str:
    """
    Fetches the transcript of a YouTube video and converts it into a subtitle-formatted string.

    Args:
        video_id (str): The unique YouTube video ID.

    Returns:
        str: The subtitles generated from the video's transcript.
    """
    return fetch_transcript_cached(video_id)

In [27]:
summary_instructions = """
You're a helpful assistant that helps answer user questions
about YouTube videos
"""

tools = [
    function_tool(fetch_youtube_transcript)
]

youtube_assistant = Agent(
    name='youtube_assistant',
    tools=tools,
    instructions=summary_instructions,
    model='gpt-4o-mini'
)

In [28]:
runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=youtube_assistant
)

In [29]:
await runner.run();

Chat ended.
